## Hybrid Search - Weighted Linear Vector Fusion (Convex Combination)


Status: Standard for Single-Pass Native Indexing.

Role: Popularized by managed vector databases (such as Pinecone, Qdrant, and Weaviate), this method allows single-pass hybrid lookups by taking a user-defined $\alpha$ weight factor. It scales dense embedding vectors and sparse term-frequency vectors before dot-product scoring, avoiding the need to execute two separate sub-queries.

General Application in a RAG Pipeline : 
$$\text{Hybrid Search (RRF or Weighted)} \longrightarrow \text{Cross-Encoder Reranking} \longrightarrow \text{MMR / Diversity Filtering} \longrightarrow \text{LLM Generation}$$

In [8]:
import os
import uuid
from dotenv import load_dotenv
from langchain_ollama import OllamaEmbeddings, ChatOllama
from pinecone import Pinecone, ServerlessSpec
from pinecone_text.sparse import BM25Encoder # it uses TF-IDF under the hood for sparse matrix 

In [9]:
load_dotenv()
pinecone_api_key = os.getenv('PINECONE_API_KEY')

In [10]:
index_name = "hybrid-search-langchain-pinecone"

# Initialize Pinecone
pc = Pinecone(api_key=pinecone_api_key)

# Create index if it doesn't exist
if index_name not in pc.list_indexes().names():
    pc.create_index(
        index_name, 
        dimension=384,  # dimension for dense vector
        metric="dotproduct",  # sparse values supported for dotproduct
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    ) 

index_obj = pc.Index(index_name)

In [11]:
# Dense Vector and Sparse matrix
embedding = OllamaEmbeddings(model="qwen3-embedding:4b", dimensions=384)
encoder = BM25Encoder().default()

In [12]:
texts = [
    "In 2023, I visited Paris",
    "In 2022, I visited New York",
    "In 2021, I visited New Orleans",
]

# Dense embeddings
dense_vectors = embedding.embed_documents(texts)

# Sparse embeddings (BM25)
sparse_vectors = encoder.encode_documents(texts)

vectors = []

for text, dense, sparse in zip(texts, dense_vectors, sparse_vectors):
    vectors.append(
        {
            "id": str(uuid.uuid4()),
            "values": dense,
            "sparse_values": sparse,
            "metadata": {
                "text": text
            }
        }
    )

index_obj.upsert(vectors=vectors)

{'upserted_count': 3}

### Applying the $\alpha$ weight

Pinecone's `dotproduct` metric combines dense and sparse scores automatically, but *how much* each side contributes still needs to be set explicitly — that's the $\alpha$ this technique is named for.

`hybrid_scale` implements the convex combination: `alpha * dense_vector + (1 - alpha) * sparse_vector`.
`alpha = 1.0` → pure dense (semantic) search. `alpha = 0.0` → pure sparse (keyword) search. `alpha = 0.5` → equal blend.

In [ ]:
def hybrid_scale(dense, sparse, alpha: float):
    """Convex combination of dense and sparse vectors.

    scaled_dense = alpha * dense
    scaled_sparse = (1 - alpha) * sparse

    Args:
        dense: list[float] - dense embedding vector
        sparse: dict with 'indices' and 'values' - BM25 sparse vector
        alpha: float in [0, 1]. 1.0 = dense only, 0.0 = sparse only
    """
    if not 0 <= alpha <= 1:
        raise ValueError("alpha must be between 0 and 1")

    hsparse = {
        "indices": sparse["indices"],
        "values": [v * (1 - alpha) for v in sparse["values"]],
    }
    hdense = [v * alpha for v in dense]
    return hdense, hsparse

In [ ]:
alpha = 0.7  # lean toward dense/semantic; try 0.3 to favor keyword matches

query = "What city did I visit last?"

dense_query = embedding.embed_query(query)
sparse_query = encoder.encode_queries(query)

# Apply the weighted linear fusion before querying
scaled_dense_query, scaled_sparse_query = hybrid_scale(dense_query, sparse_query, alpha)

results = index_obj.query(
    vector=scaled_dense_query,
    sparse_vector=scaled_sparse_query,
    top_k=3,
    include_metadata=True
)

for match in results["matches"]:
    print(match["metadata"]["text"])
    print(match["score"])
    print("-" * 50)

In 2022, I visited New York
0.965649366
--------------------------------------------------
In 2023, I visited Paris
0.949121356
--------------------------------------------------
In 2021, I visited New Orleans
0.928804398
--------------------------------------------------
